In [4]:
import pandas as pd
import numpy as np
import holidays


# 데이터 불러오기
df = pd.read_csv("비어플/F_merged (1).csv")

# 날짜 변환
df["date"] = pd.to_datetime(df["date"])

# 날짜 정렬
df = df.sort_values("date").reset_index(drop=True)


# 날짜 변수 생성
us_holidays = holidays.US()

# 월요일 
df["is_monday"] = (df["date"].dt.weekday == 0).astype(int)

# 금요일 
df["is_friday"] = (df["date"].dt.weekday == 4).astype(int)

# 거래일 여부
df["is_trading_day"] = (
    (df["date"].dt.weekday < 5) & (~df["date"].isin(us_holidays))
).astype(int)


# 주간 / 월간 변수 forward fill로 채우기
weekly_vars = [
    "OilInventories",
    "OilProduction"
]

monthly_vars = [
    "RealInterestRate",
    "CPI",
    "IndustryProduction",
    "CPE",
    "OPECProduction",
    "FedFundsRate"
]

df[weekly_vars + monthly_vars] = df[weekly_vars + monthly_vars].ffill()

# =========================
# 추가 Feature 생성

# 유가 수익률
df["oil_return"] = df["OilPrice"].pct_change(fill_method=None)

# 하루 전 유가 수익률
df["oil_return_lag1"] = df["oil_return"].shift(1)

# 1주 전 유가 수익률
df["oil_return_lag5"] = df["oil_return"].shift(5)

# 5일 변동성
df["oil_volatility_5"] = df["oil_return"].rolling(5).std()

# 재고 변화
df["inventory_change"] = df["OilInventories"].diff()

# 달러지수 변화율
df["dollar_return"] = df["DollarIndex"].pct_change(fill_method=None)

# VIX 변화
df["vix_change"] = df["VIX"].diff()



df.to_csv("model_data_final_ffill.csv", index=False)
print("\n저장 완료: model_data_final_ffill.csv")


저장 완료: model_data_final_ffill.csv


In [2]:
df.head(10)

,date,OilPrice,RealInterestRate,CPI,DollarIndex,VIX,IndustryProduction,CPE,OilInventories,OPECProduction,...,is_monday,is_friday,is_trading_day,oil_return,oil_return_lag1,oil_return_lag5,oil_volatility_5,inventory_change,dollar_return,vix_change
0,2006-01-01,NaN,2.064842,199.3,NaN,NaN,98.1999,NaN,NaN,33237.829,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-01-02,NaN,2.064842,199.3,101.4155,NaN,98.1999,NaN,NaN,33237.829,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-03,63.11,2.064842,199.3,100.7558,11.14,98.1999,NaN,NaN,33237.829,...,0,0,1,NaN,NaN,NaN,NaN,NaN,-0.006505,NaN
3,2006-01-04,63.41,2.064842,199.3,100.2288,11.37,98.1999,NaN,NaN,33237.829,...,0,0,1,0.004754,NaN,NaN,NaN,NaN,-0.005230,0.23
4,2006-01-05,62.81,2.064842,199.3,100.2992,11.31,98.1999,NaN,NaN,33237.829,...,0,0,1,-0.009462,0.004754,NaN,NaN,NaN,0.000702,-0.06
5,2006-01-06,64.21,2.064842,199.3,100.0241,11.00,98.1999,NaN,302584.0,33237.829,...,0,1,1,0.022289,-0.009462,NaN,NaN,NaN,-0.002743,-0.31
6,2006-01-09,63.56,2.064842,199.3,100.1794,11.13,98.1999,NaN,302584.0,33237.829,...,1,0,1,-0.010123,0.022289,NaN,NaN,0.0,0.001553,0.13
7,2006-01-10,63.41,2.064842,199.3,100.1436,10.86,98.1999,NaN,302584.0,33237.829,...,0,0,1,-0.002360,-0.010123,NaN,0.013340,0.0,-0.000357,-0.27
8,2006-01-11,63.91,2.064842,199.3,99.8710,10.94,98.1999,NaN,302584.0,33237.829,...,0,0,1,0.007885,-0.002360,0.004754,0.013629,0.0,-0.002722,0.08
9,2006-01-12,63.96,2.064842,199.3,100.0643,11.20,98.1999,NaN,302584.0,33237.829,...,0,0,1,0.000782,0.007885,-0.009462,0.012241,0.0,0.001935,0.26
